# Nemotron-3-Nano-30B LoRA — Submission Demo

Downloads the pre-trained LoRA adapter from Hugging Face Hub and writes it to
`/kaggle/working` for competition submission.

**No base model loading required** — the competition evaluator loads the base model
and adapter separately. This notebook only needs to produce the adapter files.

Training was done off-Kaggle on a GB10 (DGX Spark). See the
[prize eligibility notebook](https://www.kaggle.com/code/gdataranger/nemotron-3-nano-30b-lora-reasoning-challenge)
for full methodology.


## 1. Download adapter from Hugging Face Hub


In [ ]:
import os, shutil
from huggingface_hub import snapshot_download

ADAPTER_REPO = "marksusol/nemotron-nano-30b-lora-reasoning"
OUTPUT_DIR   = "/kaggle/working"
HF_TOKEN     = os.environ.get("HF_TOKEN")  # optional — repo is public

print(f"Downloading adapter from {ADAPTER_REPO} ...")
adapter_path = snapshot_download(
    repo_id=ADAPTER_REPO,
    repo_type="model",
    token=HF_TOKEN,
    ignore_patterns=["*.git*", "*.md"],
)
print(f"Downloaded to: {adapter_path}")


## 2. Copy adapter files to /kaggle/working


In [ ]:
# Copy required adapter and tokenizer files to OUTPUT_DIR
REQUIRED = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "chat_template.jinja",
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
for fname in REQUIRED:
    src = os.path.join(adapter_path, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(OUTPUT_DIR, fname))
        size = os.path.getsize(os.path.join(OUTPUT_DIR, fname))
        print(f"  {fname}  ({size:,} bytes)")
    else:
        print(f"  MISSING: {fname}")

print(f"\nAdapter ready in {OUTPUT_DIR}")


## 3. Verify output


In [ ]:
import json as _json

# Confirm adapter_config.json is valid and LoRA rank <= 32
config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
with open(config_path) as f:
    config = _json.load(f)

r = config.get("r", "?")
base = config.get("base_model_name_or_path", "?")
print(f"base_model: {base}")
print(f"lora_r:     {r}")
assert isinstance(r, int) and r <= 32, f"LoRA rank {r} exceeds competition limit of 32"
print("Competition constraints: OK")

print("\nFiles in /kaggle/working:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f}  ({size:,} bytes)")
